# Darknode AI — train a security LM from scratch (Colab)

No base model, no API. This notebook trains **Darknode AI**'s own tokenizer,
transformer and weights from random initialization, end to end, on a GPU runtime.

**Runtime → Change runtime type → GPU (T4 is enough).** Then **Runtime → Run all**.

> A from-scratch model at this scale learns domain vocabulary and house-style,
> not large-model reasoning. Add corpus + steps to scale it up.

## 1. GPU check

In [ ]:
!nvidia-smi -L || echo 'No GPU — set Runtime type to GPU for real training.'

## 2. Get the code + install
This repo is private — set a `GITHUB_TOKEN` in the next cell, or make the repo public first.

In [ ]:
# The repo is PRIVATE. Choose one:
#  A) make it public on GitHub, then this clone works as-is, or
#  B) set GITHUB_TOKEN below to a token with 'repo' read scope.
GITHUB_TOKEN = ''  # e.g. 'ghp_...'; leave '' if the repo is public
REPO = 'Darknode-Official/darknode-ai'
url = f'https://{GITHUB_TOKEN}@github.com/{REPO}.git' if GITHUB_TOKEN else f'https://github.com/{REPO}.git'
import os
if not os.path.isdir('darknode-ai'):
    rc = os.system(f'git clone {url} darknode-ai')
    assert rc == 0, 'clone failed — set GITHUB_TOKEN or make the repo public'
%cd darknode-ai
!pip install -q -e . 2>&1 | tail -1

## 3. (Optional) Mount Google Drive for checkpoints
Checkpoints survive a disconnect if written to Drive.

In [ ]:
USE_DRIVE = False  # set True to persist checkpoints to Drive
OUT_DIR = 'runs/darknode-small'
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/darknode-ai/runs/darknode-small'
print('checkpoints ->', OUT_DIR)

## 4. Build the corpus (rights-clean, self-authored + synthetic)

In [ ]:
!darknode-ai corpus --out data/corpus --synthetic 4000

## 5. Train the tokenizer (byte-level BPE, from scratch)

In [ ]:
!darknode-ai tokenizer --corpus data/corpus --out runs/tokenizer.json --vocab-size 8192

## 6. Prepare the dataset (redact → dedup → pack + version)

In [ ]:
!darknode-ai prepare --manifest data/manifest.json --tokenizer runs/tokenizer.json --out data/prepared
import json; print(json.load(open('data/prepared/meta.json')))

## 7. Train from scratch
`small` preset ≈ 15–35M params. Adjust `--max-steps` for longer runs.

In [ ]:
!darknode-ai train --preset colab_t4 --data-dir data/prepared \
    --tokenizer runs/tokenizer.json --out-dir "$OUT_DIR" --max-steps 6000

## 8. Plot the loss curve

In [ ]:
import json, matplotlib.pyplot as plt, os
recs=[json.loads(l) for l in open(os.path.join(OUT_DIR,'metrics.jsonl'))]
steps=[r['step'] for r in recs if 'loss' in r]; loss=[r['loss'] for r in recs if 'loss' in r]
plt.plot(steps,loss); plt.xlabel('step'); plt.ylabel('train loss'); plt.title('Darknode AI training'); plt.grid(True); plt.show()

## 9. Evaluate (perplexity + behavioural probes)

In [ ]:
!darknode-ai eval --ckpt "$OUT_DIR/best.pt" --tokenizer runs/tokenizer.json

## 10. Sample

In [ ]:
!darknode-ai sample --ckpt "$OUT_DIR/best.pt" --tokenizer runs/tokenizer.json \
    --prompt $'<|user|> Triage: many failed logins then one success for svc_backup.\n<|assistant|>\n'

## 11. Register the version (dataset + metrics + rollback record)

In [ ]:
!darknode-ai register --ckpt "$OUT_DIR/best.pt" --tokenizer runs/tokenizer.json \
    --version v0.1.0 --data-dir data/prepared
import json; print(json.load(open('runs/registry.json')))